# Fill-Level Forecasting Model Training
**Purpose**: Train and tune regression models (L2 Ridge regression, Random Forest, and Gradient Boosting) to forecast receptacle fill levels at a 3-hour lead horizon.

### Model Architectures:
1. **Baseline**: Historical Lag-1 Persistence Regressor ($\hat{F}_{t+3} = F_t$).
2. **Linear Benchmark**: L2-Regularized Ridge Regression ($\alpha = 1.0$).
3. **Random Forest Regressor**: 100 Trees, Max Depth = 12, Min Samples Split = 4.
4. **Gradient Boosting Regressor**: 100 Estimators, Learning Rate = 0.05, Max Depth = 5.

In [1]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

df_model = pd.read_csv('../data/processed/feature_engineered_bins.csv')
feature_cols = [
    'fill_percentage', 'distance_cm', 'total_weight_kg',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend',
    'fill_lag_1h', 'fill_lag_2h', 'fill_velocity_1h', 'weight_lag_1h'
]
target = 'target_fill_lead_3h'

# Chronological partition (80% train, 20% test)
split = int(len(df_model) * 0.8)
X_train, y_train = df_model.iloc[:split][feature_cols], df_model.iloc[:split][target]
X_test, y_test = df_model.iloc[split:][feature_cols], df_model.iloc[split:][target]

print(f"Training observations: {len(X_train)} | Test observations: {len(X_test)}")

# Train Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train, y_train)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_split=4, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Models trained successfully.")

Training observations: 1716 | Test observations: 429


Models trained successfully.


In [2]:
# Compare feature importance rankings
imp_df = pd.DataFrame({
    'Random_Forest': rf.feature_importances_,
    'Gradient_Boosting': gb.feature_importances_
}, index=feature_cols).sort_values(by='Random_Forest', ascending=False)

print("Feature Importance Distribution:")
print(imp_df.to_string())

Feature Importance Distribution:
                  Random_Forest  Gradient_Boosting
fill_percentage        0.843285           0.853467
fill_velocity_1h       0.044739           0.054361
hour_cos               0.023097           0.014816
total_weight_kg        0.021756           0.018144
fill_lag_1h            0.020199           0.027444
fill_lag_2h            0.016368           0.006393
distance_cm            0.010892           0.012685
weight_lag_1h          0.008579           0.004571
day_sin                0.005001           0.002036
hour_sin               0.002624           0.001953
is_weekend             0.002585           0.001856
day_cos                0.000875           0.002274


### Empirical Findings:
- Prior fill percentage ($F_t$) and autoregressive lag ($F_{t-1}$) contribute over 60% of total split gain.
- Cumulative deposited mass ($M_{t-1}$) and fill velocity account for the remaining non-linear variation, capturing density shifts across diverse waste types.